# 03 — Error analysis

Run **after** training (on Kaggle, or locally if checkpoints are downloaded).
Produces the paper's qualitative evidence:

1. Confusion matrices (flat 38-class; species L1; a per-species disease head).
2. Most-confused class pairs.
3. Per-class F1 table (where imbalance bites).
4. **Grad-CAM** on misclassified images — background-bias evidence.
5. Error-propagation summary for the hierarchy.

Metrics policy: macro F1 is primary; accuracy shown only as a caveated secondary.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loaders import build_dataset, build_loader
from src.data.splits import LabelMaps
from src.evaluation.metrics import compute_metrics, confusion, per_class_table
from src.evaluation.error_analysis import (
    plot_confusion_matrix, most_confused_pairs, find_misclassified, gradcam_overlay,
)
from src.models.factory import build_model
from src.utils.config import load_config
from src.utils.seed import seed_everything

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'plantvillage dataset' / 'color'
SPLITS_DIR = PROJECT_ROOT / 'data' / 'splits'
maps = LabelMaps.from_json(SPLITS_DIR / 'label_maps.json')

## Load a trained flat model

Point `EXP` at the experiment directory written by `train_flat.py`.

In [ ]:
cfg = load_config(PROJECT_ROOT / 'configs' / 'resnet50_flat.yaml')
EXP = PROJECT_ROOT / 'experiments' / cfg.experiment.name
model = build_model(dict(cfg.model.to_dict()), cfg.task.num_classes)
ckpt = torch.load(EXP / 'best.pth', map_location=device)
model.load_state_dict(ckpt['model_state']); model.to(device).eval()
print('loaded', EXP / 'best.pth', '| best', ckpt.get('monitor'), '=', round(ckpt.get('score', 0), 4))

## Collect test predictions

In [ ]:
test_ds = build_dataset(data_root=DATA_ROOT, splits_dir=SPLITS_DIR, split='test',
                        mode='flat', img_size=cfg.data.img_size)
loader = build_loader(test_ds, batch_size=64, is_train=False, num_workers=2)

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in loader:
        logits = model(x.to(device))
        y_true.append(y.numpy()); y_pred.append(logits.argmax(1).cpu().numpy())
y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred)

res = compute_metrics(y_true, y_pred, num_classes=maps.num_classes)
print('macro F1 :', round(res.macro_f1, 4), '(primary)')
print('weighted :', round(res.weighted_f1, 4))
print('bal acc  :', round(res.balanced_accuracy, 4))
print('accuracy :', round(res.accuracy, 4), '(secondary; over-credits majority classes)')

## Confusion matrix (38-class) + most-confused pairs

In [ ]:
names = [maps.id_to_class[i] for i in range(maps.num_classes)]
cm = confusion(y_true, y_pred, num_classes=maps.num_classes)
plot_confusion_matrix(cm, names, normalize=True,
                      title='Flat 38-class confusion (row-normalized = recall)')
plt.show()
most_confused_pairs(cm, names, top_k=15)

## Per-class F1 (worst classes first)

In [ ]:
per_class_table(res, names).head(15)

## Grad-CAM on misclassified examples (background-bias check)

If the heatmap concentrates on the uniform background rather than the leaf lesion,
the model is exploiting the dataset shortcut. This is the key limitation figure.

In [ ]:
wrong = find_misclassified(y_true, y_pred, max_per_pair=1)[:8]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.ravel(), wrong):
    img, true_label = test_ds[idx]
    overlay, _ = gradcam_overlay(model, img, target_class=int(y_pred[idx]))
    ax.imshow(overlay); ax.axis('off')
    ax.set_title(f"T:{maps.id_to_class[true_label].split('___')[-1]}\n"
                 f"P:{maps.id_to_class[int(y_pred[idx])].split('___')[-1]}", fontsize=7)
fig.suptitle('Grad-CAM on misclassified test images (predicted-class attribution)')
plt.tight_layout(); plt.show()

## Hierarchical error propagation (optional)

Requires a trained hierarchy; see `scripts/evaluate_pipeline.py` for the full run.
The summary dict it writes (`test_metrics.json`) contains
`share_of_errors_due_to_species` — the headline error-propagation number.

In [ ]:
import json
hier_metrics = PROJECT_ROOT / 'experiments' / 'resnet50_hierarchical' / 'test_metrics.json'
if hier_metrics.exists():
    print(json.dumps(json.loads(hier_metrics.read_text()), indent=2))
else:
    print('Run scripts/evaluate_pipeline.py first to produce', hier_metrics)